# 04. Sanghun Han - Bi-Encoder, Quantization, and Final Benchmarking

This notebook implements Sanghun Han's part of the Quora Question Pairs final project.

Main goals:
- Load and preprocess the Quora duplicate-question dataset.
- Train a lightweight Sentence-BERT style Bi-Encoder using a shared question encoder.
- Apply PyTorch dynamic quantization for CPU inference.
- Benchmark the original and quantized Bi-Encoder.
- Merge reusable teammate benchmark results when available.
- Save all generated CSV, Markdown, model, and plot outputs under `outputs/`.


## 1. Current Repository Structure

Inspected on 2026-06-01 from the repository root.

```text
.
|-- AGENT.md
|-- GOAL.md
|-- 03_siamese_lstm_jun_data_only.ipynb
|-- 04_biencoder_quantization_sanghun.ipynb  # this notebook
|-- splits/                                  # downloaded shared preprocessing split indices
|   |-- train_idx.npy
|   |-- val_idx.npy
|   `-- test_idx.npy
`-- outputs/
    |-- results/
    |   `-- siamese_lstm_results.csv
    `-- jun/
        |-- tables/*.csv
        `-- metadata/*.json, README_outputs.md
```

Important rule from `AGENT.md`: do not delete or rewrite teammates' notebooks. This notebook only adds Sanghun's work and writes new outputs under `outputs/`.


## 2. Existing Files and Reusable Outputs

Existing project files:
- `AGENT.md`: project instructions and repository rules.
- `GOAL.md`: Sanghun Han objective and deliverables.
- `03_siamese_lstm_jun_data_only.ipynb`: teammate notebook for Word2Vec + Siamese LSTM.
- `outputs/results/siamese_lstm_results.csv`: one-row benchmark summary for Jun's model.
- `outputs/jun/tables/val_split_text.csv` and `outputs/jun/tables/test_split_text.csv`: reusable validation/test split text.
- `outputs/jun/tables/test_predictions.csv`, `threshold_search_validation.csv`, `metrics_summary.csv`, and related CSVs: reusable analysis tables.
- `outputs/jun/metadata/*.json`: reusable run/config summaries.

Reusable outputs for final benchmarking:
- `outputs/results/siamese_lstm_results.csv` can be merged directly into `outputs/benchmark_results.csv`.
- The preprocessing repository `https://github.com/hykiop63/CSE36301_ML_final.git` defines the team split indices under `splits/train_idx.npy`, `splits/val_idx.npy`, and `splits/test_idx.npy`. This notebook can download those shared indices and apply them to local `data/train.csv`.
- If `outputs/for_bert/train.csv`, `val.csv`, and `test.csv` already exist from `01_preprocessing.ipynb`, this notebook loads them directly. Those files use columns `question1`, `question2`, and `label`.
- Jun's `outputs/results/siamese_lstm_results.csv` can still be merged directly into the final benchmark table.

Required dataset location if not already present:
- Put Kaggle Quora Question Pairs `train.csv` or `train.csv.zip` under `data/`. GitHub contains the split index files, but not the raw Kaggle CSV.


## 3. Exact Implementation Plan

1. Create reproducible project paths and random seeds.
2. Load `outputs/for_bert/train.csv`, `val.csv`, and `test.csv` from `01_preprocessing.ipynb` when available. If they are missing, download shared `splits/*.npy` from the preprocessing GitHub repo and apply them to local `data/train.csv`.
3. Preserve the preprocessing repo's BERT-style minimal text columns, then create Sanghun-specific word-level cleaned columns for the lightweight Bi-Encoder vocabulary.
4. Save Sanghun-readable split tables to `outputs/sanghun/tables/` and recreate `outputs/for_bert/*.csv` when using raw data plus shared split indices.
5. Build a beginner-friendly word-level vocabulary from the training split.
6. Encode each question independently so the same encoder processes `question1` and `question2`.
7. Train a lightweight PyTorch Bi-Encoder:
   - shared embedding layer
   - mean pooling over non-padding tokens
   - small projection MLP
   - pair classifier using `[q1, q2, |q1-q2|, q1*q2, cosine]`
8. Search the validation threshold that maximizes F1.
9. Evaluate the original model on the test split.
10. Apply and compare two quantization strategies: `nn.Linear`-only dynamic quantization, and improved embedding-int8 plus linear dynamic quantization.
11. Evaluate each quantized model on CPU with the same test split and threshold search.
12. Benchmark accuracy, F1, precision, recall, log loss, latency, model size, parameter count/state tensor count, and RSS memory.
13. Save Sanghun outputs:
    - `outputs/results/biencoder_quantization_sanghun_results.csv`
    - `outputs/benchmark_results.csv`
    - `outputs/accuracy_latency_tradeoff.png`
    - `outputs/accuracy_model_size_tradeoff.png`
    - `outputs/model_size_comparison.png`
    - `outputs/final_benchmark_summary.md`
14. Merge any existing benchmark result CSVs under `outputs/results/` into the final benchmark table without modifying teammate files.


In [ ]:
# =========================
# 0. Imports, paths, and reproducibility
# =========================

from __future__ import annotations

from pathlib import Path
from collections import Counter
import json
import math
import os
import random
import re
import time
import zipfile
import urllib.request
import copy

# Keep Matplotlib cache inside the repo outputs folder on systems where ~/.matplotlib is not writable.
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / "outputs" / ".matplotlib"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, log_loss
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Required for dynamic quantization on some CPU builds where the default is "none".
if torch.backends.quantized.engine == "none" and torch.backends.quantized.supported_engines:
    preferred_engines = ["fbgemm", "qnnpack", "x86"]
    for engine in preferred_engines:
        if engine in torch.backends.quantized.supported_engines:
            torch.backends.quantized.engine = engine
            break

try:
    import psutil
except Exception:
    psutil = None

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
SPLIT_DIR = ROOT / "splits"
BERT_DIR = OUTPUT_DIR / "for_bert"
SANGHUN_DIR = OUTPUT_DIR / "sanghun"
TABLE_DIR = SANGHUN_DIR / "tables"
MODEL_DIR = SANGHUN_DIR / "models"
META_DIR = SANGHUN_DIR / "metadata"
RESULT_DIR = OUTPUT_DIR / "results"

for directory in [DATA_DIR, OUTPUT_DIR, SPLIT_DIR, BERT_DIR, SANGHUN_DIR, TABLE_DIR, MODEL_DIR, META_DIR, RESULT_DIR, Path(os.environ["MPLCONFIGDIR"] )]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = RANDOM_STATE) -> None:
    """Make the run as reproducible as practical for a class project notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)
print("Repository root:", ROOT)
print("Device:", DEVICE)


In [ ]:
# =========================
# 1. Configuration
# =========================

# For the first runnable version, use a bounded sample so the pipeline finishes on CPU.
# For final reported results, set QUICK_RUN=False and rerun all cells.
QUICK_RUN = False

# Compact model default: smaller vocab/embedding makes quantization visibly effective.
CONFIG = {
    "random_state": RANDOM_STATE,
    "quick_run": QUICK_RUN,
    "quick_train_rows": 30000,
    "quick_val_rows": 5000,
    "quick_test_rows": 5000,
    "max_vocab_size": 20000,
    "min_token_count": 2,
    "max_sequence_length": 40,
    "embedding_dim": 64,
    "projection_dim": 64,
    "dropout": 0.20,
    "batch_size": 512,
    "epochs": 3,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "patience": 2,
    "num_workers": 0,
    "benchmark_batches": 30,
}

with open(META_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

CONFIG


In [ ]:
# =========================
# 2. Dataset loading and preprocessing
# =========================

PREPROCESSING_REPO_RAW = "https://raw.githubusercontent.com/hykiop63/CSE36301_ML_final/main"
SHARED_SPLIT_URLS = {
    "train": f"{PREPROCESSING_REPO_RAW}/splits/train_idx.npy",
    "val": f"{PREPROCESSING_REPO_RAW}/splits/val_idx.npy",
    "test": f"{PREPROCESSING_REPO_RAW}/splits/test_idx.npy",
}

CONTRACTIONS = {
    "what's": "what is", "what're": "what are", "who's": "who is",
    "where's": "where is", "when's": "when is", "why's": "why is",
    "how's": "how is", "it's": "it is", "he's": "he is",
    "she's": "she is", "that's": "that is", "there's": "there is",
    "they're": "they are", "we're": "we are", "you're": "you are",
    "i've": "i have", "you've": "you have", "we've": "we have",
    "they've": "they have", "i'd": "i would", "you'd": "you would",
    "he'd": "he would", "she'd": "she would", "we'd": "we would",
    "they'd": "they would", "i'll": "i will", "you'll": "you will",
    "he'll": "he will", "she'll": "she will", "we'll": "we will",
    "they'll": "they will", "i'm": "i am", "can't": "cannot",
    "couldn't": "could not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not",
    "wasn't": "was not", "weren't": "were not", "won't": "will not",
    "wouldn't": "would not", "haven't": "have not", "hasn't": "has not",
    "hadn't": "had not", "should've": "should have",
    "would've": "would have", "could've": "could have",
}

def clean_for_bert(text: object) -> str:
    """Same minimal BERT-style cleaning used by 01_preprocessing.ipynb."""
    if not isinstance(text, str) or text == "":
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_text(text: object) -> str:
    """Word-level cleaning for this lightweight Bi-Encoder vocabulary."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    for src, dst in CONTRACTIONS.items():
        text = text.replace(src, dst)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def standardize_pair_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize either raw Quora or outputs/for_bert CSVs to this notebook's schema."""
    out = df.copy()
    if "label" in out.columns and "is_duplicate" not in out.columns:
        out = out.rename(columns={"label": "is_duplicate"})
    required = ["question1", "question2", "is_duplicate"]
    missing = [col for col in required if col not in out.columns]
    if missing:
        raise ValueError(f"Input data is missing required columns: {missing}")
    out["question1"] = out["question1"].fillna("").astype(str)
    out["question2"] = out["question2"].fillna("").astype(str)
    out["is_duplicate"] = out["is_duplicate"].astype(int)
    out["q1_bert"] = out["question1"].map(clean_for_bert)
    out["q2_bert"] = out["question2"].map(clean_for_bert)
    out["q1_clean"] = out["question1"].map(clean_text)
    out["q2_clean"] = out["question2"].map(clean_text)
    return out.reset_index(drop=True)

def find_train_csv() -> Path:
    """Find or extract Kaggle Quora train.csv."""
    candidates = [DATA_DIR / "train.csv", ROOT / "train.csv", Path("/kaggle/input/quora-question-pairs/train.csv")]
    for path in candidates:
        if path.exists():
            return path

    zip_candidates = [DATA_DIR / "train.csv.zip", DATA_DIR / "quora-question-pairs.zip", ROOT / "train.csv.zip"]
    for zip_path in zip_candidates:
        if zip_path.exists():
            print(f"Extracting {zip_path} into {DATA_DIR}")
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(DATA_DIR)
            if (DATA_DIR / "train.csv").exists():
                return DATA_DIR / "train.csv"

    raise FileNotFoundError(
        "Could not find Quora train.csv. Put Kaggle train.csv or train.csv.zip under data/. "
        "The GitHub preprocessing repo provides split indices, not the raw Kaggle CSV."
    )

def load_preprocessed_for_bert() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame] | None:
    """Use 01_preprocessing.ipynb outputs directly when available."""
    paths = {name: BERT_DIR / f"{name}.csv" for name in ["train", "val", "test"]}
    if all(path.exists() for path in paths.values()):
        print("Loading preprocessing outputs from outputs/for_bert/*.csv")
        return tuple(standardize_pair_frame(pd.read_csv(paths[name])) for name in ["train", "val", "test"])
    return None

def ensure_shared_split_indices() -> bool:
    """Download team split indices from the preprocessing GitHub repo when missing."""
    paths = {name: SPLIT_DIR / f"{name}_idx.npy" for name in ["train", "val", "test"]}
    if all(path.exists() for path in paths.values()):
        return True
    print("Shared split files are missing. Trying to download them from preprocessing GitHub repo...")
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    try:
        for name, url in SHARED_SPLIT_URLS.items():
            target = paths[name]
            if not target.exists():
                urllib.request.urlretrieve(url, target)
                print(f"Downloaded {target}")
        return all(path.exists() for path in paths.values())
    except Exception as exc:
        print("Could not download shared split indices:", exc)
        return False

def create_for_bert_from_raw_with_shared_splits(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame] | None:
    """Recreate preprocessing repo's outputs/for_bert CSVs using shared split indices."""
    if not ensure_shared_split_indices():
        return None

    df = standardize_pair_frame(raw_df)
    split_arrays = {name: np.load(SPLIT_DIR / f"{name}_idx.npy") for name in ["train", "val", "test"]}
    if max(int(arr.max()) for arr in split_arrays.values()) >= len(df):
        raise ValueError("Shared split indices do not match the loaded raw train.csv row count.")

    parts = []
    for name in ["train", "val", "test"]:
        part = df.iloc[split_arrays[name]].reset_index(drop=True)
        bert_out = part[["q1_bert", "q2_bert", "is_duplicate"]].copy()
        bert_out.columns = ["question1", "question2", "label"]
        bert_out.to_csv(BERT_DIR / f"{name}.csv", index=False)
        part.to_csv(TABLE_DIR / f"{name}_split_text.csv", index=False)
        parts.append(part)
        print(f"[{name}] {len(part):,} rows from shared preprocessing split")
    print("Saved recreated preprocessing-compatible CSVs under outputs/for_bert/.")
    return tuple(parts)

def create_own_splits_from_raw(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fallback only: use this when preprocessing outputs and shared split indices are unavailable."""
    df = standardize_pair_frame(raw_df)
    train_df, temp_df = train_test_split(
        df, test_size=0.20, stratify=df["is_duplicate"], random_state=RANDOM_STATE
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df["is_duplicate"], random_state=RANDOM_STATE
    )
    train_df, val_df, test_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        part.to_csv(TABLE_DIR / f"{name}_split_text.csv", index=False)
    print("WARNING: Used local stratified split fallback, not preprocessing repo shared split files.")
    return train_df, val_df, test_df

def maybe_quick_sample(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    if not QUICK_RUN:
        return train_df, val_df, test_df
    train_df = train_df.sample(min(CONFIG["quick_train_rows"], len(train_df)), random_state=RANDOM_STATE).reset_index(drop=True)
    val_df = val_df.sample(min(CONFIG["quick_val_rows"], len(val_df)), random_state=RANDOM_STATE).reset_index(drop=True)
    test_df = test_df.sample(min(CONFIG["quick_test_rows"], len(test_df)), random_state=RANDOM_STATE).reset_index(drop=True)
    return train_df, val_df, test_df

def load_or_create_splits() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Data priority: outputs/for_bert -> raw train.csv + GitHub shared splits -> fallback split."""
    preprocessed = load_preprocessed_for_bert()
    if preprocessed is not None:
        return maybe_quick_sample(*preprocessed)

    data_path = find_train_csv()
    print("Loading raw dataset:", data_path)
    raw_df = pd.read_csv(data_path)
    raw_df["question1"] = raw_df["question1"].fillna("")
    raw_df["question2"] = raw_df["question2"].fillna("")

    shared = create_for_bert_from_raw_with_shared_splits(raw_df)
    if shared is not None:
        return maybe_quick_sample(*shared)
    return maybe_quick_sample(*create_own_splits_from_raw(raw_df))

train_df, val_df, test_df = load_or_create_splits()
print("Train/val/test sizes:", len(train_df), len(val_df), len(test_df))
print("Train duplicate rate:", train_df["is_duplicate"].mean())


In [ ]:
# =========================
# 3. Vocabulary and PyTorch datasets
# =========================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_ID = 0
UNK_ID = 1

def build_vocab(texts: list[str], max_vocab_size: int, min_count: int) -> dict[str, int]:
    counter = Counter()
    for text in texts:
        counter.update(str(text).split())
    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for token, count in counter.most_common(max_vocab_size - len(vocab)):
        if count < min_count:
            break
        vocab[token] = len(vocab)
    return vocab

def encode_text(text: str, vocab: dict[str, int], max_len: int) -> list[int]:
    ids = [vocab.get(tok, UNK_ID) for tok in str(text).split()[:max_len]]
    if not ids:
        ids = [UNK_ID]
    if len(ids) < max_len:
        ids = ids + [PAD_ID] * (max_len - len(ids))
    return ids

text_col_q1 = "q1_clean" if "q1_clean" in train_df.columns else "q1_lstm"
text_col_q2 = "q2_clean" if "q2_clean" in train_df.columns else "q2_lstm"

vocab = build_vocab(
    train_df[text_col_q1].astype(str).tolist() + train_df[text_col_q2].astype(str).tolist(),
    max_vocab_size=CONFIG["max_vocab_size"],
    min_count=CONFIG["min_token_count"],
)
with open(META_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

class QuestionPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocab: dict[str, int], max_len: int):
        self.q1 = np.array([encode_text(x, vocab, max_len) for x in df[text_col_q1].astype(str)], dtype=np.int64)
        self.q2 = np.array([encode_text(x, vocab, max_len) for x in df[text_col_q2].astype(str)], dtype=np.int64)
        self.y = df["is_duplicate"].astype(np.float32).values

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return (
            torch.tensor(self.q1[idx], dtype=torch.long),
            torch.tensor(self.q2[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32),
        )

train_ds = QuestionPairDataset(train_df, vocab, CONFIG["max_sequence_length"])
val_ds = QuestionPairDataset(val_df, vocab, CONFIG["max_sequence_length"])
test_ds = QuestionPairDataset(test_df, vocab, CONFIG["max_sequence_length"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

print("Vocabulary size:", len(vocab))


In [ ]:
# =========================
# 4. Bi-Encoder model
# =========================


class Int8Embedding(nn.Module):
    """Inference-only int8 embedding table with one symmetric scale.

    Dynamic quantization does not shrink nn.Embedding in this project setup,
    and the embedding table dominates model size. This module stores the
    embedding weights as int8 buffers and dequantizes only the selected token
    vectors during lookup.
    """
    def __init__(self, qweight: torch.Tensor, scale: float, padding_idx: int = PAD_ID):
        super().__init__()
        self.register_buffer("qweight", qweight.to(torch.int8))
        self.register_buffer("scale", torch.tensor(float(scale), dtype=torch.float32))
        self.padding_idx = padding_idx
        self.embedding_dim = int(qweight.shape[1])
        self.num_embeddings = int(qweight.shape[0])

    @classmethod
    def from_float(cls, embedding: nn.Embedding) -> "Int8Embedding":
        weight = embedding.weight.detach().cpu()
        max_abs = weight.abs().max().clamp(min=1e-8)
        scale = float(max_abs / 127.0)
        qweight = torch.round(weight / scale).clamp(-127, 127).to(torch.int8)
        if embedding.padding_idx is not None:
            qweight[embedding.padding_idx].zero_()
        return cls(qweight, scale, embedding.padding_idx if embedding.padding_idx is not None else PAD_ID)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        selected = F.embedding(token_ids, self.qweight, padding_idx=self.padding_idx)
        return selected.float() * self.scale


def replace_embedding_with_int8(model_obj: nn.Module) -> nn.Module:
    """Return an inference copy whose embedding table is stored as int8."""
    converted = copy.deepcopy(model_obj).cpu().eval()
    converted.encoder.embedding = Int8Embedding.from_float(converted.encoder.embedding)
    return converted

class MeanPoolingEncoder(nn.Module):
    """Shared sentence encoder: embedding lookup -> masked mean pooling -> projection MLP."""
    def __init__(self, vocab_size: int, embedding_dim: int, projection_dim: int, dropout: float):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim, projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim, projection_dim),
        )

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        mask = (token_ids != PAD_ID).float().unsqueeze(-1)
        emb = self.embedding(token_ids)
        summed = (emb * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1.0)
        pooled = summed / lengths
        return self.projection(pooled)

class BiEncoderDuplicateModel(nn.Module):
    """Encode each question separately, then classify the pair with lightweight features."""
    def __init__(self, vocab_size: int, embedding_dim: int, projection_dim: int, dropout: float):
        super().__init__()
        self.encoder = MeanPoolingEncoder(vocab_size, embedding_dim, projection_dim, dropout)
        pair_feature_dim = projection_dim * 4 + 1
        self.classifier = nn.Sequential(
            nn.Linear(pair_feature_dim, projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim, 1),
        )

    def forward(self, q1_ids: torch.Tensor, q2_ids: torch.Tensor) -> torch.Tensor:
        z1 = self.encoder(q1_ids)
        z2 = self.encoder(q2_ids)
        cosine = F.cosine_similarity(z1, z2).unsqueeze(1)
        features = torch.cat([z1, z2, torch.abs(z1 - z2), z1 * z2, cosine], dim=1)
        return self.classifier(features).squeeze(1)

model = BiEncoderDuplicateModel(
    vocab_size=len(vocab),
    embedding_dim=CONFIG["embedding_dim"],
    projection_dim=CONFIG["projection_dim"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def count_state_values(model: nn.Module) -> int:
    return sum(t.numel() for t in model.state_dict().values() if torch.is_tensor(t))

print(model)
print("Parameter count:", count_parameters(model))


In [ ]:
# =========================
# 5. Training, inference, and threshold helpers
# =========================

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    total_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()
    for q1, q2, y in loader:
        q1, q2, y = q1.to(DEVICE), q2.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(q1, q2)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * len(y)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def predict_proba(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs, labels = [], []
    for q1, q2, y in loader:
        q1, q2 = q1.to(device), q2.to(device)
        logits = model(q1, q2)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels).astype(int)

def find_best_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, pd.DataFrame]:
    rows = []
    for threshold in np.arange(0.20, 0.81, 0.01):
        pred = (y_prob >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            "accuracy": accuracy_score(y_true, pred),
            "f1": f1_score(y_true, pred, zero_division=0),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
        })
    df = pd.DataFrame(rows)
    best = df.sort_values(["f1", "accuracy"], ascending=False).iloc[0]
    return float(best["threshold"]), df

def evaluate_predictions(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict[str, float]:
    pred = (y_prob >= threshold).astype(int)
    return {
        "Test Accuracy": float(accuracy_score(y_true, pred)),
        "Test F1": float(f1_score(y_true, pred, zero_division=0)),
        "Test Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Test Recall": float(recall_score(y_true, pred, zero_division=0)),
        "Test Log Loss": float(log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6))),
    }

def save_model_size(model: nn.Module, path: Path) -> float:
    torch.save(model.state_dict(), path)
    return path.stat().st_size / (1024 * 1024)

@torch.no_grad()
def benchmark_latency(model: nn.Module, loader: DataLoader, device: torch.device, max_batches: int) -> tuple[float, float]:
    """Return total measured seconds and milliseconds per sample."""
    model.eval()
    total_samples = 0
    start = time.perf_counter()
    for batch_idx, (q1, q2, _) in enumerate(loader):
        if batch_idx >= max_batches:
            break
        q1, q2 = q1.to(device), q2.to(device)
        _ = model(q1, q2)
        total_samples += q1.size(0)
    elapsed = time.perf_counter() - start
    return elapsed, (elapsed * 1000.0 / max(total_samples, 1))

def memory_rss_mb() -> float | None:
    if psutil is None:
        return None
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)


In [ ]:
# =========================
# 6. Train original Bi-Encoder
# =========================

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0
history = []

training_start = time.perf_counter()
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_prob, val_y = predict_proba(model, val_loader, DEVICE)
    best_threshold, threshold_df = find_best_threshold(val_y, val_prob)
    val_metrics = evaluate_predictions(val_y, val_prob, best_threshold)
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "best_val_threshold": best_threshold,
        "val_accuracy": val_metrics["Test Accuracy"],
        "val_f1": val_metrics["Test F1"],
    }
    history.append(row)
    print(row)

    if val_metrics["Test F1"] > best_val_f1:
        best_val_f1 = val_metrics["Test F1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIG["patience"]:
            print("Early stopping triggered.")
            break

training_time_sec = time.perf_counter() - training_start
if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df.to_csv(TABLE_DIR / "biencoder_training_history.csv", index=False)
print("Training time sec:", training_time_sec)


In [ ]:
# =========================
# 7. Evaluate original and quantized Bi-Encoder
# =========================

def build_result_row(model_name: str, model_obj: nn.Module, device: torch.device, model_size_mb: float,
                     training_time: float | None, threshold_type: str,
                     role: str, embedding_source: str, epochs_run: int | None = None) -> dict[str, object]:
    val_prob, val_y = predict_proba(model_obj, val_loader, device)
    threshold, threshold_df = find_best_threshold(val_y, val_prob)
    threshold_df.to_csv(TABLE_DIR / f"{model_name.lower().replace(' ', '_').replace('+', 'plus')}_threshold_search.csv", index=False)

    test_prob, test_y = predict_proba(model_obj, test_loader, device)
    metrics = evaluate_predictions(test_y, test_prob, threshold)
    inference_sec, inference_ms = benchmark_latency(model_obj, test_loader, device, CONFIG["benchmark_batches"])

    pred_df = pd.DataFrame({
        "y_true": test_y.astype(int),
        "prob_duplicate": test_prob.astype(float),
        "pred_label": (test_prob >= threshold).astype(int),
    })
    pred_df.to_csv(TABLE_DIR / f"{model_name.lower().replace(' ', '_').replace('+', 'plus')}_test_predictions.csv", index=False)

    row = {
        "Model": model_name,
        "Owner": "Sanghun Han",
        "Role": role,
        "Embedding Source": embedding_source,
        "Official Threshold Type": threshold_type,
        "Best Threshold": threshold,
        "Accuracy Threshold": threshold,
        "Train Samples": int(len(train_df)),
        "Validation Samples": int(len(val_df)),
        "Test Samples": int(len(test_df)),
        **metrics,
        "Training Time sec": None if training_time is None else float(training_time),
        "Inference Time sec": float(inference_sec),
        "Inference ms/sample": float(inference_ms),
        "Model Size MB": float(model_size_mb),
        "Encoder Size MB": None,
        "Parameter Count": int(count_parameters(model_obj)),
        "Encoder Parameter Count": int(count_parameters(model_obj.encoder)) if hasattr(model_obj, "encoder") else None,
        "State Tensor Count": int(count_state_values(model_obj)),
        "Memory RSS MB": memory_rss_mb(),
        "Max Sequence Length": int(CONFIG["max_sequence_length"]),
        "Vocabulary Size": int(len(vocab)),
        "Embedding Dim": int(CONFIG["embedding_dim"]),
        "LSTM Units": None,
        "Batch Size": int(CONFIG["batch_size"]),
        "Epochs Run": int(len(history_df) if epochs_run is None else epochs_run),
        "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only",
    }
    return row

# Original model benchmark.
original_model_path = MODEL_DIR / "biencoder_original_state_dict.pt"
original_size_mb = save_model_size(model.cpu(), original_model_path)
model = model.to(torch.device("cpu"))
cpu_device = torch.device("cpu")
original_row = build_result_row(
    "Bi-Encoder",
    model,
    cpu_device,
    original_size_mb,
    training_time_sec,
    "validation_f1_best",
    "Compact Bi-Encoder duplicate-question detector",
    "trainable_word_embeddings_mean_pooling_compact",
)

# Dynamic quantization, linear layers only. This is kept as an ablation to show
# why the first quantization attempt barely changed model size.
linear_quantized_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(model).cpu().eval(),
    {nn.Linear},
    dtype=torch.qint8,
)
linear_quantized_model_path = MODEL_DIR / "biencoder_linear_only_quantized_state_dict.pt"
linear_quantized_size_mb = save_model_size(linear_quantized_model, linear_quantized_model_path)
linear_quantized_row = build_result_row(
    "Linear-Only Quantized Bi-Encoder",
    linear_quantized_model,
    cpu_device,
    linear_quantized_size_mb,
    None,
    "validation_f1_best_after_linear_quantization",
    "Dynamic quantization applied only to nn.Linear layers",
    "trainable_word_embeddings_mean_pooling_compact_linear_int8",
)

# Improved quantization: store the dominant embedding table as int8, then apply
# dynamic quantization to Linear layers. This directly targets the model-size
# bottleneck measured above.
embedding_int8_model = replace_embedding_with_int8(model)
embedding_int8_linear_quantized_model = torch.quantization.quantize_dynamic(
    embedding_int8_model,
    {nn.Linear},
    dtype=torch.qint8,
)
embedding_int8_model_path = MODEL_DIR / "biencoder_embedding_int8_linear_quantized_state_dict.pt"
embedding_int8_size_mb = save_model_size(embedding_int8_linear_quantized_model, embedding_int8_model_path)
embedding_int8_row = build_result_row(
    "Embedding+Linear Int8 Bi-Encoder",
    embedding_int8_linear_quantized_model,
    cpu_device,
    embedding_int8_size_mb,
    None,
    "validation_f1_best_after_embedding_and_linear_quantization",
    "Embedding table stored as int8 plus dynamic quantization for nn.Linear",
    "trainable_word_embeddings_mean_pooling_compact_embedding_int8_linear_int8",
)

sanghun_results_df = pd.DataFrame([original_row, linear_quantized_row, embedding_int8_row])
sanghun_results_path = RESULT_DIR / "biencoder_quantization_sanghun_results.csv"
sanghun_results_df.to_csv(sanghun_results_path, index=False)
sanghun_results_df


In [ ]:
# =========================
# 7b. Long training with early stopping
# =========================

# Final-model experiment: allow many epochs, but select the checkpoint with the
# best validation F1 and stop when validation F1 no longer improves.
RUN_EARLY_STOPPING_EXPERIMENT = True
MAX_EARLY_STOPPING_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5
MIN_F1_IMPROVEMENT = 1e-4

if RUN_EARLY_STOPPING_EXPERIMENT:
    set_seed(RANDOM_STATE)
    es_model = BiEncoderDuplicateModel(
        vocab_size=len(vocab),
        embedding_dim=CONFIG["embedding_dim"],
        projection_dim=CONFIG["projection_dim"],
        dropout=CONFIG["dropout"],
    ).to(DEVICE)
    es_optimizer = torch.optim.AdamW(
        es_model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )

    es_best_val_f1 = -1.0
    es_best_state = None
    es_best_epoch = 0
    es_epochs_without_improvement = 0
    es_history = []
    es_training_start = time.perf_counter()

    for epoch in range(1, MAX_EARLY_STOPPING_EPOCHS + 1):
        train_loss = train_one_epoch(es_model, train_loader, es_optimizer)
        val_prob, val_y = predict_proba(es_model, val_loader, DEVICE)
        best_threshold, _ = find_best_threshold(val_y, val_prob)
        val_metrics = evaluate_predictions(val_y, val_prob, best_threshold)
        current_val_f1 = float(val_metrics["Test F1"])
        improved = current_val_f1 > (es_best_val_f1 + MIN_F1_IMPROVEMENT)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "best_val_threshold": best_threshold,
            "val_accuracy": val_metrics["Test Accuracy"],
            "val_f1": current_val_f1,
            "improved": bool(improved),
            "best_epoch_so_far": int(epoch if improved else es_best_epoch),
        }
        es_history.append(row)
        print(row)

        if improved:
            es_best_val_f1 = current_val_f1
            es_best_epoch = epoch
            es_best_state = {k: v.detach().cpu().clone() for k, v in es_model.state_dict().items()}
            es_epochs_without_improvement = 0
        else:
            es_epochs_without_improvement += 1
            if es_epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch}; best epoch was {es_best_epoch}.")
                break

    es_training_time_sec = time.perf_counter() - es_training_start
    if es_best_state is not None:
        es_model.load_state_dict(es_best_state)

    es_history_df = pd.DataFrame(es_history)
    es_history_df.to_csv(TABLE_DIR / "biencoder_early_stopping_training_history.csv", index=False)

    es_model_path = MODEL_DIR / "biencoder_early_stopping_original_state_dict.pt"
    es_original_size_mb = save_model_size(es_model.cpu(), es_model_path)
    es_model = es_model.to(torch.device("cpu"))

    es_original_row = build_result_row(
        "Bi-Encoder Early Stopping",
        es_model,
        cpu_device,
        es_original_size_mb,
        es_training_time_sec,
        "validation_f1_best_early_stopping",
        f"Compact Bi-Encoder trained up to {MAX_EARLY_STOPPING_EPOCHS} epochs with early stopping",
        "trainable_word_embeddings_mean_pooling_compact_early_stopping",
        int(es_best_epoch),
    )
    es_original_row["Max Epochs"] = int(MAX_EARLY_STOPPING_EPOCHS)
    es_original_row["Early Stopping Patience"] = int(EARLY_STOPPING_PATIENCE)
    es_original_row["Actual Epochs Run"] = int(len(es_history_df))

    es_embedding_int8_model = replace_embedding_with_int8(es_model)
    es_embedding_int8_linear_quantized_model = torch.quantization.quantize_dynamic(
        es_embedding_int8_model,
        {nn.Linear},
        dtype=torch.qint8,
    )
    es_embedding_int8_path = MODEL_DIR / "biencoder_early_stopping_embedding_int8_linear_quantized_state_dict.pt"
    es_embedding_int8_size_mb = save_model_size(es_embedding_int8_linear_quantized_model, es_embedding_int8_path)
    es_embedding_int8_row = build_result_row(
        "Embedding+Linear Int8 Bi-Encoder Early Stopping",
        es_embedding_int8_linear_quantized_model,
        cpu_device,
        es_embedding_int8_size_mb,
        None,
        "validation_f1_best_early_stopping_after_embedding_and_linear_quantization",
        "Early-stopped compact Bi-Encoder with int8 embedding table and dynamic quantized Linear layers",
        "trainable_word_embeddings_mean_pooling_compact_early_stopping_embedding_int8_linear_int8",
        int(es_best_epoch),
    )
    es_embedding_int8_row["Max Epochs"] = int(MAX_EARLY_STOPPING_EPOCHS)
    es_embedding_int8_row["Early Stopping Patience"] = int(EARLY_STOPPING_PATIENCE)
    es_embedding_int8_row["Actual Epochs Run"] = int(len(es_history_df))

    early_stopping_results_df = pd.DataFrame([es_original_row, es_embedding_int8_row])
    early_stopping_results_path = RESULT_DIR / "biencoder_early_stopping_sanghun_results.csv"
    early_stopping_results_df.to_csv(early_stopping_results_path, index=False)

    print("Saved early stopping results:", early_stopping_results_path)
    display_cols = [
        "Model", "Test Accuracy", "Test F1", "Inference ms/sample", "Model Size MB",
        "Epochs Run", "Actual Epochs Run", "Training Time sec",
    ]
    print(early_stopping_results_df[display_cols].to_string(index=False))
else:
    print("Early stopping experiment skipped.")


In [ ]:
# =========================
# 8. Final benchmark table and report-focused plots
# =========================

def load_all_result_csvs(result_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(result_dir.glob("*.csv")):
        try:
            df = pd.read_csv(path)
            df["Source File"] = str(path.relative_to(ROOT))
            frames.append(df)
        except Exception as exc:
            print("Skipping unreadable result CSV:", path, exc)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)

def dataframe_to_markdown_table(df: pd.DataFrame) -> str:
    """Small Markdown table writer that does not require the optional tabulate package."""
    if df.empty:
        return "No benchmark rows are available yet."
    safe_df = df.fillna("").astype(str)
    header = "| " + " | ".join(safe_df.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(safe_df.columns)) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in safe_df.to_numpy()]
    return "\n".join([header, separator] + rows)

benchmark_df = load_all_result_csvs(RESULT_DIR)
benchmark_path = OUTPUT_DIR / "benchmark_results.csv"
benchmark_df.to_csv(benchmark_path, index=False)

# Final report plots focus on the real final comparison, not every epoch ablation.
final_names = [
    "Bi-Encoder Early Stopping",
    "Embedding+Linear Int8 Bi-Encoder Early Stopping",
    "Word2Vec + Siamese LSTM",
]
final_df = benchmark_df[benchmark_df["Model"].isin(final_names)].copy()
final_df["Model"] = pd.Categorical(final_df["Model"], categories=final_names, ordered=True)
final_df = final_df.sort_values("Model")
short_labels = {
    "Bi-Encoder Early Stopping": "Bi-Encoder\nfloat",
    "Embedding+Linear Int8 Bi-Encoder Early Stopping": "Bi-Encoder\nint8",
    "Word2Vec + Siamese LSTM": "Siamese\nLSTM",
}
final_df["Short Label"] = final_df["Model"].astype(str).map(short_labels)
final_df.to_csv(TABLE_DIR / "final_models_for_visualization.csv", index=False)

PLOT_DIR = SANGHUN_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
colors = {
    "Bi-Encoder\nfloat": "#4C78A8",
    "Bi-Encoder\nint8": "#F58518",
    "Siamese\nLSTM": "#54A24B",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

def save_to_both(fig, filename: str) -> None:
    fig.savefig(OUTPUT_DIR / filename)
    fig.savefig(PLOT_DIR / filename)
    plt.close(fig)

def save_bar_metric(metric: str, ylabel: str, title: str, filename: str, value_fmt: str = "{:.3f}", ylim: tuple[float, float] | None = None) -> None:
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    labels = final_df["Short Label"].tolist()
    vals = final_df[metric].astype(float).to_numpy()
    bars = ax.bar(labels, vals, color=[colors[x] for x in labels], width=0.62)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim:
        ax.set_ylim(*ylim)
    ax.grid(axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), value_fmt.format(val), ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    save_to_both(fig, filename)

save_bar_metric("Test F1", "Test F1", "Final Model F1 Comparison", "final_f1_comparison.png", "{:.3f}", (0.60, 0.80))
save_bar_metric("Test Accuracy", "Test Accuracy", "Final Model Accuracy Comparison", "final_accuracy_comparison.png", "{:.3f}", (0.70, 0.84))
save_bar_metric("Model Size MB", "Model size (MB)", "Final Model Size Comparison", "final_model_size_comparison.png", "{:.2f}")

fig, ax = plt.subplots(figsize=(7.2, 4.8))
for _, row in final_df.iterrows():
    label = row["Short Label"]
    ax.scatter(row["Model Size MB"], row["Test F1"], s=120, color=colors[label], edgecolor="white", linewidth=1.2)
    ax.annotate(label.replace("\n", " "), (row["Model Size MB"], row["Test F1"]), xytext=(7, 4), textcoords="offset points", fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Model size (MB, log scale)")
ax.set_ylabel("Test F1")
ax.set_title("Accuracy-Efficiency Trade-off: F1 vs Model Size")
ax.grid(alpha=0.25, which="both")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save_to_both(fig, "final_f1_model_size_tradeoff.png")

fig, ax = plt.subplots(figsize=(7.2, 4.8))
for _, row in final_df.iterrows():
    label = row["Short Label"]
    ax.scatter(row["Inference ms/sample"], row["Test F1"], s=120, color=colors[label], edgecolor="white", linewidth=1.2)
    ax.annotate(label.replace("\n", " "), (row["Inference ms/sample"], row["Test F1"]), xytext=(7, 4), textcoords="offset points", fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Inference latency (ms/sample, log scale)")
ax.set_ylabel("Test F1")
ax.set_title("Accuracy-Efficiency Trade-off: F1 vs Latency")
ax.grid(alpha=0.25, which="both")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save_to_both(fig, "final_f1_latency_tradeoff.png")

bi = final_df[final_df["Model"].astype(str).str.contains("Bi-Encoder")].copy()
fig, ax1 = plt.subplots(figsize=(7.2, 4.6))
labels = bi["Short Label"].tolist()
x = np.arange(len(labels))
width = 0.36
size_bars = ax1.bar(x - width / 2, bi["Model Size MB"].astype(float), width, label="Size MB", color="#F58518")
ax1.set_ylabel("Model size (MB)")
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.grid(axis="y", alpha=0.25)
ax2 = ax1.twinx()
f1_bars = ax2.bar(x + width / 2, bi["Test F1"].astype(float), width, label="F1", color="#4C78A8")
ax2.set_ylabel("Test F1")
ax2.set_ylim(0.74, 0.79)
for bar in size_bars:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
for bar in f1_bars:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
ax1.set_title("Bi-Encoder Compression Effect")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
fig.tight_layout()
save_to_both(fig, "biencoder_compression_effect.png")

traditional_plot_df = traditional_df.sort_values("Test F1", ascending=True) if len(traditional_df) > 0 else pd.DataFrame()
if len(traditional_plot_df) > 0:
    fig, ax = plt.subplots(figsize=(8.4, 5.2))
    ax.barh(traditional_plot_df["Model"], traditional_plot_df["Test F1"], color="#B279A2")
    ax.set_xlabel("Test F1")
    ax.set_title("Traditional ML Baselines")
    ax.grid(axis="x", alpha=0.25)
    for i, value in enumerate(traditional_plot_df["Test F1"]):
        ax.text(value, i, f" {value:.3f}", va="center", fontsize=9)
    fig.tight_layout()
    save_to_both(fig, "traditional_ml_f1_comparison.png")

hist_path = TABLE_DIR / "biencoder_early_stopping_training_history.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    best_epoch = int(hist.loc[hist["val_f1"].idxmax(), "epoch"])
    best_val_f1 = float(hist["val_f1"].max())
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    ax.plot(hist["epoch"], hist["val_f1"], color="#4C78A8", linewidth=2.0, label="Validation F1")
    ax.plot(hist["epoch"], hist["val_accuracy"], color="#72B7B2", linewidth=1.8, label="Validation accuracy")
    ax.axvline(best_epoch, color="#E45756", linestyle="--", linewidth=1.5, label=f"Best epoch {best_epoch}")
    ax.scatter([best_epoch], [best_val_f1], color="#E45756", s=70, zorder=3)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation metric")
    ax.set_title("Early Stopping Curve")
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    save_to_both(fig, "early_stopping_validation_curve.png")

summary_cols = [
    "Model", "Owner", "Test Accuracy", "Test F1", "Inference ms/sample", "Model Size MB",
    "Best Threshold", "Train Samples", "Validation Samples", "Test Samples", "Source File",
]
existing_cols = [col for col in summary_cols if col in benchmark_df.columns]
summary_md = "# Final Benchmark Summary\n\n"
summary_md += "Generated by `04_biencoder_quantization_sanghun.ipynb`.\n\n"
summary_md += dataframe_to_markdown_table(benchmark_df[existing_cols])
summary_md += "\n\nNotes:\n"
summary_md += "- Final report plots compare the best Traditional ML baseline, Siamese LSTM, early-stopped float Bi-Encoder, and early-stopped int8 Bi-Encoder.\n"
summary_md += "- 3-epoch and 6-epoch rows remain in the CSV as training-length ablations, but they are not the primary final comparison.\n"
summary_md += "- Linear-only quantization is included as an ablation; it barely shrinks models whose embedding table dominates size.\n"
summary_md += "- The improved quantized Bi-Encoder stores the embedding table as int8 and applies dynamic quantization to `nn.Linear` layers.\n"
(OUTPUT_DIR / "final_benchmark_summary.md").write_text(summary_md, encoding="utf-8")

print("Saved report-focused outputs:")
for name in [
    "final_f1_comparison.png",
    "final_accuracy_comparison.png",
    "final_model_size_comparison.png",
    "final_f1_model_size_tradeoff.png",
    "final_f1_latency_tradeoff.png",
    "biencoder_compression_effect.png",
    "early_stopping_validation_curve.png",
]:
    print("-", OUTPUT_DIR / name)
print("-", TABLE_DIR / "final_models_for_visualization.csv")
print("-", OUTPUT_DIR / "final_benchmark_summary.md")
benchmark_df
